In [24]:
import numpy as np
import matplotlib.pyplot as plt
from core.file_manager import preprocess_file_manager

In [25]:
orginal_data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessed_steps = {
    'nifi_to_raw' : {'start': '0_nifty', 'end': '1_raw'},
    'filling_anatomy_gaps' : {'start': '1_raw', 'end': '2_anatomy_gap_filled'},
    'cropping' : {'start': '2_anatomy_gap_filled', 'end': '3_cropped'}
}

crop_size = (160,160,24)

target_spacing=(0.8, 0.8, 3.5)

In [26]:
file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)
patients = file_manager.get_file_names()

In [27]:
step = '0_nifty'

In [28]:
from collections import Counter

def get_spacings_set(file_manager, step, patients):
    spacings = []
    for patient in patients:
        data = file_manager.load_file_NifTy(step, patient)
        spacing = data['anatomy'].GetSpacing()
        spacings.append(spacing)
    spacing_counter = Counter(spacings)
    print("Unique spacings:", len(spacing_counter))
    for spacing, cnt in sorted(spacing_counter.items()):
        print(f"spacing {spacing} -> {cnt}")
    print("\nAll unique spacing values:", set(spacings))
    spacing_set = set(spacings)
    return spacing_set

# spacing_set = get_spacings_set(file_manager, step, patients)

# print(spacing_set)

In [29]:
problematic_ones = {}
for patient in patients:
    spacings = []
    data = file_manager.load_file_NifTy(step, patient)
    for channel in data:
        spacing = data['anatomy'].GetSpacing()
        spacings.append(spacing)
    if len(set(spacings)) > 1:
        problematic_ones[patient] = spacings

print("Patients with inconsistent spacings across channels:")
for patient, spacings in problematic_ones.items():
    print(f"{patient}: {spacings}")
    

Patients with inconsistent spacings across channels:


In [30]:
data = file_manager.load_file_NifTy(step, patient)
type(data['anatomy'])

SimpleITK.SimpleITK.Image

some preprocessing stuff

In [31]:
import SimpleITK as sitk
import numpy as np


from core.transformers.resample_transformer import resample_transformer

In [33]:
test_patients = ['3322']
test_Start = '0_nifty'
test_End = 'TEST_spacinged'

for patient in test_patients:
    data = file_manager.load_file_NifTy(test_Start, patient)
    data['anatomy'] = resample_transformer.resample_nii(None,data['anatomy'], target_spacing=target_spacing, is_label=False)
    data['t2'] = resample_transformer.resample_nii(None,data['t2'], target_spacing=target_spacing, is_label=False)
    data['dwi'] = resample_transformer.resample_nii(None,data['dwi'], target_spacing=target_spacing, is_label=False)
    data['adc'] = resample_transformer.resample_nii(None,data['adc'], target_spacing=target_spacing, is_label=False)
    file_manager.save_file_Nifty(test_End, patient, data)
    